# 03 — Modélisation bayésienne multi-sorties + Calibration

Objectif: entraîner un modèle bayésien, produire des incertitudes, et **calibrer** les intervalles (temperature scaling / isotonic).

## 0) Données d'entraînement
Pour la démo, on utilise un jeu **synthétique contrôlé** (même seeds que la CLI).

In [ ]:

import os, sys, json, numpy as np
sys.path.append(os.path.join("..","src"))
from raman_ai.models.train import train_bayesian_multioutput
from raman_ai.eval.calibration import interval_coverage, ece_regression, isotonic_sigma_mapping

np.random.seed(7)
N, D, K = 256, 24, 4
X = np.random.randn(N, D)
W = np.random.randn(D, K)
Y = X @ W + 0.12*np.random.randn(N, K)
Xtr, Ytr = X[:200], Y[:200]
Xva, Yva = X[200:], Y[200:]

out_dir = train_bayesian_multioutput(Xtr, Ytr, X_val=Xva, Y_val=Yva, alpha=1.3, beta=25.0)
out_dir


## 1) Prédictions + métriques brutes

In [ ]:

with open(os.path.join("..","models","bayes_multi","val_pred.json"), "r") as fh:
    pred = json.load(fh)
mu  = np.array(pred["mu"])
std = np.array(pred["std"])

mae = float(np.mean(np.abs(mu - Yva)))
ece = ece_regression(Yva.flatten(), mu.flatten(), std.flatten(), n_bins=12)
mae, ece


## 2) Couvertures nominales vs empiriques

In [ ]:

confs = [0.5, 0.8, 0.9, 0.95]
cov_emp = [interval_coverage(Yva.flatten(), mu.flatten(), std.flatten(), conf=c).empirical for c in confs]
list(zip(confs, cov_emp))


## 3) Calibration par **temperature scaling** (1 paramètre)

In [ ]:

def fit_temperature_scaling(y_true, mu, std, targets=(0.5,0.8,0.9,0.95)):
    y = y_true.flatten(); m = mu.flatten(); s0 = std.flatten()
    best_s, best_loss = 1.0, 1e9
    for s in np.linspace(0.3, 3.0, 108):
        covs = [interval_coverage(y, m, s0*s, conf=t).empirical for t in targets]
        loss = float(np.mean((np.array(covs) - np.array(targets))**2))
        if loss < best_loss:
            best_s, best_loss = float(s), loss
    return best_s

s_opt = fit_temperature_scaling(Yva, mu, std)
std_t = std * s_opt
cov_t = [interval_coverage(Yva.flatten(), mu.flatten(), std_t.flatten(), conf=c).empirical for c in confs]
ece_t = ece_regression(Yva.flatten(), mu.flatten(), std_t.flatten(), n_bins=12)
s_opt, cov_t, ece_t


## 4) Calibration **isotonique** (mapping monotone sur σ)

In [ ]:

f_iso = isotonic_sigma_mapping(Yva, mu, std, n_quantiles=15)
std_i = f_iso(std)
cov_i = [interval_coverage(Yva.flatten(), mu.flatten(), std_i.flatten(), conf=c).empirical for c in confs]
ece_i = ece_regression(Yva.flatten(), mu.flatten(), std_i.flatten(), n_bins=12)
cov_i, ece_i


## 5) Figures (reliability plots)
Les figures sont sauvegardées dans `plots/` et injectées par `plots/figures.tex` dans le PDF IEEE.

In [ ]:

import matplotlib.pyplot as plt, os
plots_dir = os.path.join("..","plots")
os.makedirs(plots_dir, exist_ok=True)

def plot_calib(nominal, empirical, out_png, title):
    plt.figure()
    plt.plot(nominal, empirical, marker="o")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("Nominal")
    plt.ylabel("Empirical")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

plot_calib(confs, cov_emp, os.path.join(plots_dir, "uncertainty_calibration_raw.png"), "Calibration (raw)")
plot_calib(confs, cov_t,   os.path.join(plots_dir, "uncertainty_calibration_temp.png"), "Calibration (temp)")
plot_calib(confs, cov_i,   os.path.join(plots_dir, "uncertainty_calibration_iso.png"),  "Calibration (isotonic)")
os.listdir(plots_dir)[:6]


## 6) Export des métriques et injection LaTeX

In [ ]:

import json
metrics = {
    "val_mae": float(np.mean(np.abs(mu - Yva))),
    "raw": {"nominal": confs, "empirical": cov_emp},
    "temp": {"scale": float(s_opt), "empirical": cov_t, "ece": float(ece_t)},
    "isotonic": {"empirical": cov_i, "ece": float(ece_i)}
}
with open(os.path.join("..","metrics","summary_modeling.json"), "w") as fh:
    json.dump(metrics, fh, indent=2)

# figures.tex est déjà géré par le repo ; on s'assure simplement que les PNG existent
metrics
